# HQIV CMB: Planck epoch → now

Run the full-sky HQIV CMB pipeline and compare with Planck.

- **Evolver**: `HQIVUniverseEvolver(nside=1024).run_from_T_Pl_to_now()`
- **Output**: `T_map_muK` (µK), `sigma8`, C_ℓ (TT, EE, TE, BB)
- **Secondaries**: lapse-compressed galaxy accelerated motion (ISW/Rees–Sciama)

Requires `pip install pyhqiv[cosmology]` for healpy map.

In [ ]:
from pyhqiv.cosmology import HQIVUniverseEvolver

evolver = HQIVUniverseEvolver(nside=64)  # or 1024 for full res
result = evolver.run_from_T_Pl_to_now()

# Print σ₈ and summary (visible in notebook)
print("=== HQIV CMB run_from_T_Pl_to_now ===")
print(f"  σ₈ = {result['sigma8']:.4f}")
if result.get("Omega_k_true") is not None:
    print(f"  Ω_k^true = {result['Omega_k_true']:.4f}")
print(f"  C_ℓ: ell in [2, {result['ell'][-1]:.0f}], len = {len(result['ell'])}")
if result.get("T_map_muK") is not None:
    t = result["T_map_muK"]
    print(f"  T_map_muK: npix = {len(t)}, mean = {t.mean():.2f} µK, std = {t.std():.2f} µK")
else:
    print("  T_map_muK: None (install pyhqiv[cosmology] for healpy)")

In [ ]:
import healpy as hp

if result.get("T_map_muK") is not None:
    hp.mollview(result["T_map_muK"], title="HQIV CMB from Planck epoch to now")
    hp.graticule()
else:
    print("No T_map_muK (install pyhqiv[cosmology] for healpy map)")

## Multipole chart: ℓ(ℓ+1)C_ℓ/2π vs ℓ

Plot of the CMB temperature power spectrum (D_ℓ in µK²) as a function of multipole ℓ. Acoustic peaks appear at ℓ ≈ 220, 540, 810 (flat ΛCDM); HQIV with Ω_k = 0.0098 shifts them slightly.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

ell = result["ell"]
c_tt = result["C_ell_TT"]
d_tt = ell * (ell + 1) / (2 * np.pi) * np.maximum(c_tt, 1e-30)

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.semilogy(ell, d_tt, label=f"HQIV (σ₈ = {result['sigma8']:.3f})")
# Add Planck 2018 C_ℓ here (e.g. from CDN or data file) for comparison
ax.set_xlabel(r"Multipole $\ell$")
ax.set_ylabel(r"$\ell(\ell+1)C_\ell/2\pi$ [µK²]")
ax.set_title("HQIV CMB multipole — T_Pl to now")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(2, ell[-1])
plt.tight_layout()
plt.show()